## The physical properties of post-mass-transfer binaries

R. Seeburger1,5,? , H.-W. Rix1, K. El-Badry1,2, J. Muller-Horn1,5 , A. J. Dimoff J. Henneco6,3 , and J. I. Villaseñor

2026

https://www.aanda.org/articles/aa/pdf/2026/01/aa53916-25.pdf

Six binary stellar systems, originally proposed as possible star-black hole binaries on the basis of radial velocities from Gaia’s third data release, but soon recognised as likely post-mass-transfer binary systems with stripped companions


In [1]:
import numpy as np
import json
import pandas as pd

import astropy.units as u
from astroquery.vizier import Vizier
import re

# Ensure project root is on sys.path so `import paths` finds the top-level paths.py
import os, sys
from pathlib import Path
proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import DATA_DIR, RAW_JSON_DIR

In [2]:
# ----------------------------------------------------------------------------
# Six post-MT binaries from Seeburger et al. 2026 (A&A 705, A146): cool, bloated,
# very low-mass stripped stars (proto-He WDs) with rapidly rotating A/F-type
# accretors. Originally Gaia DR3 star-BH candidates (El-Badry & Rix 2022,
# MNRAS 515, 1266 = EB+22).
#
# Positions and orbits: Gaia DR3 (gaia_source + SB1 solution in nss_two_body_orbit).
# Masses and Teff: Seeburger+26 Table 3 (disentangling + SED; not purely dynamical
# -> quality flags indirect_M1 / indirect_M2).
# ----------------------------------------------------------------------------
from astroquery.gaia import Gaia

# Seeburger+26 Table 3: donor = stripped star (-> M2), accretor = A/F star (-> M1)
SEEBURGER26 = {
    2933630927108779776: {"short": "G-2933", "M_don": (0.27, 0.04), "M_acc": (2.2, 0.2), "Teff_don": 4750, "Teff_acc": 9500},
    2966694650501747328: {"short": "G-2966", "M_don": (0.23, 0.03), "M_acc": (2.0, 0.1), "Teff_don": 5500, "Teff_acc": 9000},
    5243109471519822720: {"short": "G-5243", "M_don": (0.28, 0.04), "M_acc": (2.0, 0.2), "Teff_don": 4750, "Teff_acc": 8750},
    5536105058044762240: {"short": "G-5536", "M_don": (0.23, 0.04), "M_acc": (1.9, 0.2), "Teff_don": 6750, "Teff_acc": 7250},
    5694373091078326784: {"short": "G-5694", "M_don": (0.24, 0.04), "M_acc": (2.1, 0.2), "Teff_don": 4750, "Teff_acc": 9250},
    6000420920026118656: {"short": "G-6000", "M_don": (0.19, 0.03), "M_acc": (2.2, 0.2), "Teff_don": 4750, "Teff_acc": 9000},
}

ids = ", ".join(str(s) for s in SEEBURGER26)
job = Gaia.launch_job(f"""
SELECT g.source_id, g.ra, g.dec, g.ra_error, g.dec_error,
       n.period, n.period_error, n.eccentricity, n.eccentricity_error
FROM gaiadr3.gaia_source AS g
JOIN gaiadr3.nss_two_body_orbit AS n USING (source_id)
WHERE g.source_id IN ({ids}) AND n.nss_solution_type = 'SB1'
""")
gaia = {int(r["source_id"]): r for r in job.get_results()}
assert len(gaia) == len(SEEBURGER26), f"expected {len(SEEBURGER26)} Gaia SB1 rows, got {len(gaia)}"

Please be advised that the system will experience intermittent service interruptions next Monday (06-07-2026), between 9:30 AM and 12:00 PM, due to scheduled hardware maintenance.


In [5]:
json_dict = []
for sid, s in SEEBURGER26.items():
    r = gaia[sid]
    acc_type = "A" if s["Teff_acc"] >= 8000 else "F"
    notes = (f"Cool, bloated, very low-mass stripped star (proto-He WD, Teff ~ {s['Teff_don']} K) with a "
             f"rapidly rotating {acc_type}-type accretor (Teff ~ {s['Teff_acc']} K); recently detached and "
             f"contracting towards the (hot) pre-He WD phase. Originally a Gaia DR3 star-BH candidate (EB+22). "
             f"Orbit from the Gaia DR3 SB1 solution; masses from disentangling + SED (Seeburger+26 Table 3). "
             f"Alias: {s['short']}.")
    if sid == 2933630927108779776:
        notes += " Also known as TYC 5954-2370-1."
    json_dict.append({
        "System Name": f"Gaia DR3 {sid}",
        "RA": round(float(r["ra"]), 7),                                  # deg
        "Dec": round(float(r["dec"]), 7),                                # deg
        "pos_err_mas": round(float(max(r["ra_error"], r["dec_error"])), 4),
        "Reference": ["2026A&A...705A.146S", "2022MNRAS.515.1266E"],
        "Notes": notes,
        "Period": [round(float(r["period_error"]), 6), round(float(r["period"]), 6), round(float(r["period_error"]), 6)],
        "Eccentricity": [round(float(r["eccentricity_error"]), 4), round(float(r["eccentricity"]), 4), round(float(r["eccentricity_error"]), 4)],
        "M1": [s["M_acc"][1], s["M_acc"][0], s["M_acc"][1]],             # accretor, Msun
        "M2": [s["M_don"][1], s["M_don"][0], s["M_don"][1]],             # stripped donor, Msun
        "Mass Function": [None, None, None],
        "M1_sin3i": [None, None, None],
        "M2_sin3i": [None, None, None],
        "system_class": "EL CVn",
        "obs_type_1": f"{acc_type}-type",
        "obs_type_2": "pre-He WD",
        "evol_type_1": "MS",
        "evol_type_2": "WD",
        "Detection Method": ["RV", "SB1"],
        "quality_flags": ["indirect_M1", "indirect_M2"],
    })

for e in json_dict:
    assert 0 <= e["RA"] < 360 and -90 <= e["Dec"] <= 90 and e["Period"][1] > 0, e

display(pd.DataFrame([{k: e[k] for k in ("System Name", "RA", "Dec", "pos_err_mas", "Period", "Eccentricity", "M1", "M2")} for e in json_dict]))

,System Name,RA,Dec,pos_err_mas,Period,Eccentricity,M1,M2
0,Gaia DR3 2933630927108779776,103.691442,-18.721649,0.0127,"[0.000664, 14.717528, 0.000664]","[0.0104, 0.0211, 0.0104]","[0.2, 2.2, 0.2]","[0.04, 0.27, 0.04]"
1,Gaia DR3 2966694650501747328,88.015265,-18.938831,0.0128,"[0.001063, 10.397987, 0.001063]","[0.0132, 0.0195, 0.0132]","[0.1, 2.0, 0.1]","[0.03, 0.23, 0.03]"
2,Gaia DR3 5243109471519822720,148.577051,-69.652663,0.0120,"[0.000521, 14.913679, 0.000521]","[0.0059, 0.0227, 0.0059]","[0.2, 2.0, 0.2]","[0.04, 0.28, 0.04]"
3,Gaia DR3 5536105058044762240,113.848267,-42.646146,0.0138,"[0.003852, 12.176559, 0.003852]","[0.0462, 0.0858, 0.0462]","[0.2, 1.9, 0.2]","[0.04, 0.23, 0.04]"
4,Gaia DR3 5694373091078326784,123.866217,-26.150417,0.0154,"[0.003997, 12.884807, 0.003997]","[0.0375, 0.0176, 0.0375]","[0.2, 2.1, 0.2]","[0.04, 0.24, 0.04]"
5,Gaia DR3 6000420920026118656,229.033335,-44.322339,0.0269,"[0.002082, 15.318045, 0.002082]","[0.0092, 0.0061, 0.0092]","[0.2, 2.2, 0.2]","[0.03, 0.19, 0.03]"


In [6]:
# Save as a raw ingest table (Combine_and_process_data.py globs raw_json/*.json)
out_path = RAW_JSON_DIR / "Seeburger26_stripped_stars.raw.json"
with open(out_path, "w") as f:
    json.dump(json_dict, f, indent=2)
print(f"Wrote {len(json_dict)} systems to {out_path}")
# -> now run code/data_processing/Combine_and_process_data.py to ingest

Wrote 6 systems to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/Seeburger26_stripped_stars.raw.json
